# Choose splits and inspect their membership

## Goal and setup
Load the pinned Premier League 2023/24 and 2024/25 publications, assemble a match
dataset, and inspect chronological and retrospective split plans. Use the existing
`misc314_py314` kernel. This notebook trains no model.

Default feature/split availability uses earlier kickoff as a retrospective proxy,
not measured result-publication time. The [guide](../docs/analytics/splits.md)
contains timing examples and equations; the [reference](../docs/analytics/splits_reference.md)
lists every setting and validation rule.

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
from xdiyo_analytics.data import load_seasons, select_stats
from xdiyo_analytics.histories import build_team_history
from xdiyo_analytics.features import IsHome, Lag, Stat, evaluate_features
from xdiyo_analytics.labels import MatchTotal, create_labels
from xdiyo_analytics.datasets import assemble_dataset
from xdiyo_analytics.splits import (
    TemporalSplit, MatchKFold, GroupKFold, CPCV,
    create_split_plan, reconstruct_paths,
)

root = Path('C:/Users/luisi/Documents/Programming/Python/xDiyo')
loaded = load_seasons(
    root / 'data/xDiyo_data', ['23_24', '24_25'], leagues='Premier_League',
    tables=['matches', 'statistics'],
    record_dir=root / 'experiment/initial_population/selections', verify_hashes=True,
)
history = build_team_history(select_stats(
    loaded, stats=[('ALL', 'Match overview', 'cornerKicks')],
))
corners = Stat('ALL', 'Match overview', 'cornerKicks')
features = evaluate_features(
    history, {'venue': IsHome(), 'previous_corners': Lag(corners)}, keyed=True,
)
labels = create_labels(history, {'corners': MatchTotal(corners)})
dataset = assemble_dataset(features, labels['corners'], layout='match')


## Steps: choose chronological membership
Start with 20 observed round blocks, hold out six at a time, and score after the
first held-out block. Earlier test rows stay held out. Keep the final short horizon.
Training eligibility can remove postponed or unavailable matches.

In [2]:
scheme = TemporalSplit(
    train_size=20, test_size=6, score_start=1, allow_partial_test=True,
)
plan = create_split_plan(dataset, scheme)
fold_counts = pd.DataFrame([
    {'fold': i, 'train': len(f.train), 'test': len(f.test), 'score': len(f.score),
     'fit_at': f.metadata['fit_at']}
    for i, f in enumerate(plan.folds)
])
fold_counts.head()


,fold,train,test,score,fit_at
0,0,198,60,50,2024-01-12 19:45:00+00:00
1,1,258,60,50,2024-03-02 15:00:00+00:00
2,2,313,60,50,2024-04-13 11:30:00+00:00
3,3,380,60,50,2024-08-16 19:00:00+00:00
4,4,440,60,50,2024-10-05 11:30:00+00:00


In [3]:
fold = plan.folds[0]
train_X = dataset.X.iloc[fold.train]
test_X = dataset.X.iloc[fold.test]
score_y = dataset.y.iloc[fold.score]
assert set(fold.train).isdisjoint(fold.test)
assert set(fold.score) <= set(fold.test)
dataset.metadata.iloc[fold.test].head()


,source_league,source_season,competition_id,season_id,event_id,kickoff_at,round,status,home_id,away_id
198,Premier_League,23_24,17,52186,11352318,2024-01-12 19:45:00+00:00,21,finished,6,72
199,Premier_League,23_24,17,52186,11352321,2024-01-13 12:30:00+00:00,21,finished,38,43
200,Premier_League,23_24,17,52186,11352331,2024-01-13 17:30:00+00:00,21,finished,39,17
201,Premier_League,23_24,17,52186,11352324,2024-01-14 14:00:00+00:00,21,finished,48,40
202,Premier_League,23_24,17,52186,11352328,2024-01-14 16:30:00+00:00,21,finished,35,33


## Compare other split families
Match/group K-fold and CPCV are retrospective. GroupKFold below holds competition-season
groups together. CPCV keeps kickoff ties together and records 15 folds and five
paths under its default configuration. Its default intervals are points at kickoff;
meaningful information-overlap purging requires justified interval timestamps.

In [4]:
match_plan = create_split_plan(dataset, MatchKFold(5, shuffle=True, random_state=41))
season_plan = create_split_plan(dataset, GroupKFold(2))
external_groups = pd.Series(
    list(zip(dataset.metadata.source_league, dataset.metadata.source_season)),
    index=dataset.metadata.index, dtype=object,
)
external_plan = create_split_plan(dataset, GroupKFold(2), groups=external_groups)
assert sum(len(f.test) for f in match_plan.folds) == len(dataset.X)

cpcv = CPCV(n_blocks=6, n_test_blocks=2)
cpcv_plan = create_split_plan(dataset, cpcv)
assert cpcv_plan.get_n_splits() == 15 and cpcv.n_paths == 5
cpcv_plan.paths[['path_id', 'block_id', 'fold_id']].head(8)


,path_id,block_id,fold_id
0,0,0,0
1,0,1,0
2,0,2,1
3,0,3,2
4,0,4,3
5,0,5,4
6,1,0,1
7,1,1,5


In [5]:
choices = {'chronological': plan, 'match_kfold': match_plan,
           'competition_season': season_plan, 'cpcv': cpcv_plan}
assert all(p.model_selector is None and p.refit_policy is None for p in choices.values())
assert all(set(f.score) <= set(f.test) and set(f.train).isdisjoint(f.test)
           for p in choices.values() for f in p.folds)
pd.DataFrame([
    {'scheme': name, 'folds': p.get_n_splits(),
     'first_train': len(p.folds[0].train), 'first_test': len(p.folds[0].test),
     'first_score': len(p.folds[0].score),
     'paths': 0 if p.paths is None else p.paths.path_id.nunique()}
    for name, p in choices.items()
])

,scheme,folds,first_train,first_test,first_score,paths
0,chronological,10,198,60,50,0
1,match_kfold,5,608,152,152,0
2,competition_season,2,380,380,380,0
3,cpcv,15,515,245,245,5


## Checks and next steps
Use original positions with `.iloc`; preserve the same dataset order when passing
`plan.split()` as a future sklearn `cv` iterable. Match tuples are identities,
not ranker group sizes. The separate scoring subset is available on each Fold.

`model_selector=None` and `refit_policy=None` are stored options only. They disable
future selection/additional scheduled refits, without removing the need for a
future initial fit in each outer fold. This module invokes none of them.

CPCV paths share observations/models. Retrospective evaluation still needs
fold-aware feature/state preparation: interval purging alone does not remove
held-out outcomes already carried into Glicko or rolling features. Statistical
reports, inner selection, strategy execution and fitting remain later layers.
See the [verification evidence](../docs/analytics/splits_check.json).